In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens, _parse_parent_annotations
    
from src.tokenizer_utils import tokenize, decode
from src.htmlLabel import simplified_to_normal_form
from src.models import get_messages
from tqdm import tqdm


/home/zagar/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose the prompting configuration

In [75]:
filename = "1989CanLII1415ONCA"
split = "test"
filepath = Path(DATA_DIR) / "original" / split / f"{filename}.html"
filepath = Path("output")  / f"{filename}_2.html"

#filepath = Path("output") / "test_qwen7b_DEC_fs6_greedy_sentence" / filename / f"{filename}_2.html"
with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()

In [76]:
### Choose the right worflow
method = "DEC3" # "AIO" | "DEC0" | "DEC1" | "DEC2" | "DEC3"

if method == "AIO":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = []
    new_labels = ["decision", "legislation", "secondary sources", "title", "citation", "source", "authors", "fragment"]

    spans_in_context = True

    prompt_filename = "allInOne_long.txt"


if method == "DEC0":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = []
    new_labels = ["decision", "legislation", "secondary sources"]

    spans_in_context = True

    prompt_filename = "decomposed0_long.txt"

if method == "DEC1":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = ["decision", "legislation", "secondary sources"]
    new_labels = ["title", "fragment"]

    spans_in_context = False


    prompt_filename = "decomposed1-3.txt"

if method == "DEC2":
    parents = ["secondary sources"]
    already_labeled_labels = ["decision", "legislation", "secondary sources", "title", "fragment"]
    new_labels = ["source", "authors"]

    spans_in_context = False

    prompt_filename = "decomposed1-3.txt"

if method == "DEC3":
    parents = ["decision", "legislation"]
    already_labeled_labels = ["decision", "legislation", "secondary sources", "title", "fragment", "source", "authors"]
    new_labels = ["citation"]

    spans_in_context = False

    prompt_filename = "decomposed1-3.txt"

#### Common Few SHot Selection

In [77]:
fewshot_method = "greedy"   # "greedy" | "random"

with open(FEWSHOT_CACHE_DIR / f"examples_{fewshot_method}.json", "r", encoding="utf-8") as f:
    fewshot_file_content = json.load(f)

fewshot_examples = [(example["example"]["input"], example["example"]["output"]) for example in fewshot_file_content["examples"]]



##### Few Shot processing step

In [78]:
nb_fewshot_examples = 6
allowed_labels = already_labeled_labels + new_labels

input_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_labels=already_labeled_labels,
    keep_attributes=["labelname"]
)

output_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_labels=already_labeled_labels + new_labels,
    keep_attributes=["labelname"]
)


# Transform the output in it simplified form
final_fewshot = []
total_output_text = ""
for example in fewshot_examples:
    input, output = example

    input_tokens = tokenize(input)
    transformed_input_tokens = prepare_label_tokens(input_tokens, input_label_config)

    output_tokens = tokenize(output)
    transformed_output_tokens = prepare_label_tokens(output_tokens, output_label_config)

    if spans_in_context:
        final_fewshot.append((decode(transformed_input_tokens), decode(transformed_output_tokens)))

    if not spans_in_context:

        total_output_text += "|||" + decode(transformed_output_tokens)

final_fewshot = final_fewshot[:nb_fewshot_examples]


if not spans_in_context:
    parents_dict = _parse_parent_annotations(total_output_text)
    for parent_name, annotations in parents_dict.items():
        if parent_name not in parents:
            continue
        for annotation in annotations:
            input = decode(prepare_label_tokens(simplified_to_normal_form(tokenize(annotation),label_type="manual_label"), input_label_config))

            if input != annotation:
                final_fewshot.append((input, annotation))

#### Common Prompt loading

In [79]:
from src.prompts.prompt_utils import build_sublabel_definitions

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

from src.prompts.sublabel_definitions import SUBLABEL_DEFINITIONS_V2


if method in ["DEC1", "DEC2", "DEC3"]:
    sublabels_str = ", ".join(new_labels)
    sublabels_definition = build_sublabel_definitions(set(new_labels) - set(parents), sublabel_definitions=SUBLABEL_DEFINITIONS_V2)

    system_prompt = system_prompt.format(
            sublabels=sublabels_str,
            sublabels_definition=sublabels_definition,
        )

system_prompt used :  decomposed1-3.txt


#### Assistant loading

In [7]:
from src.models import AssistantFactory

#gpt5_2_config= {
#        "type": "openai",
#        "model_name": "gpt-5.2",
#        "temperature": 1,
#    }

#assistant = AssistantFactory.create_from_config(gpt5_2_config)


# SaulLM-7B-Instruct
# Qwen2.5-7B-Instruct
# Qwen2.5-32B-Instruct
assistant = AssistantFactory.create("Qwen2.5-7B-Instruct")

Loading Qwen2.5-7B-Instruct from /home/zagar/scratch/Qwen2.5-7B-Instruct [fp16]


2026-06-10 06:32:21.664576: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 06:32:24.265588: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781098344.466264  980889 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781098344.548060  980889 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781098344.971636  980889 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

#### Chunk output controle

In [80]:
def process_output(generated, token_chunk, allowed_labels, assistant, with_fallback: bool = True):

    from src.output_control.processor import OutputProcessor
    from src.output_control.fallback import FallbackHandler 
    from src.output_control.verification import VerificationResult 

    controller = OutputProcessor()
    fallback_handler = FallbackHandler(processor=controller)
    
    corrected_generated_tokens, status = controller.process(
        raw_llm_output=generated,
        original_chunk=token_chunk,
        allowed_labels=allowed_labels
    )

    if status.passed:
        return corrected_generated_tokens, status

    if not with_fallback:
        return token_chunk, status
    
    
    corrected_generated_tokens, status_dict = fallback_handler.handle_failure(
        assistant=assistant,
        corrected_output=corrected_generated_tokens,
        original_chunk=token_chunk,
        initial_status=status,
        allowed_labels=allowed_labels,
        fallback_prompt_filename="fallback.txt"
    )
    # Convert dict to VerificationResult
    status = VerificationResult(
        passed=status_dict.get('passed', False),
        error_type=status_dict.get('error_type'),
        details=status_dict.get('error_details'),
        tokens=corrected_generated_tokens
    )
    
    return corrected_generated_tokens, status

### For AIO or DEC0 ONLY

#### Chunking with the chunker

In [48]:
chunker = "sentence"  # "paragraph" | "sentence"

from src.chunkers.cache import cache_exists, load_cache
from src.chunkers import ChunkerFactory

if not cache_exists(chunker, split, filename):

    # Load spaCy only if needed
    nlp = None
    if chunker == "sentence":
        import spacy
        nlp = spacy.load("en_core_web_trf")
        print("✅ Model loaded.\n")


    token_chunks = ChunkerFactory.get_chunks(
        html_content, method=chunker, split=split, filename=filename, nlp=nlp
    )

else:
    token_chunks = load_cache(chunker, split, filename)




#### Test for SaulLM

In [10]:
token_chunk = token_chunks[0]


In [11]:
user_input =  decode(token_chunk)
print("User input : ", user_input)

User input :   Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the Excise Tax Act,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the Excise Tax Act. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
Schedule III of the Excise Tax Act. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.
Member W.
Roy Hines W. Roy
Hines
Member
Robert J. Martin Robert J. Martin
Secretary UNOFFICIAL
SUMMARY Appeal
No. 2845 CAN
TRAFFIC S

In [12]:
messages = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot[:1], has_system_role=True)


In [50]:
def _format_saul_prompt(messages: list) -> str:
    """
    Formats messages for SaulLM-7B (Llama-2 based).
    System message is injected into the <<SYS>> block of the first user turn.
    """
    system_content = ""
    conversation = []
    
    for msg in messages:
        if msg["role"] == "system":
            system_content = msg["content"]
        else:
            conversation.append(msg)
    
    prompt = "<s>"
    
    for i, msg in enumerate(conversation):
        if msg["role"] == "user":
            prompt += "[INST] "
            # Inject system prompt into the first user turn only
            if i == 0 and system_content:
                prompt += f"<<SYS>>\n{system_content}\n<</SYS>>\n\n"
            prompt += f"{msg['content']} [/INST]"
        elif msg["role"] == "assistant":
            prompt += f" {msg['content']} </s><s>"
    
    return prompt

In [51]:
print(messages)


[{'role': 'system', 'content': 'Context\nIn Canadian decisions, legal sources (legislation, decisions, or secondary sources) are cited to support reasoning. References may be full citations, short forms, abbreviations, or contextual clues (e.g., "the Act").\n\nRole\nAnnotate legal source references in Canadian legal decisions.\n\nLegal Source Types\n<legislation> – Statutes, regulations, constitutions, treaties (e.g., Criminal Code, s. 8 of the Charter)\n<decision> – Court or tribunal decisions (e.g., R. v. Jordan, 2016 SCC 27)\n<secondary sources> – Scholarship, commentaries, journal articles, legal dictionaries (e.g., Driedger, Sullivan)\n\nScope\nIdentify all mentions, including: first and subsequent mentions, full/short/abbreviated citations, contextual references ("the Act"), Latin terms (ibid., supra), and footnotes.\n\nLabel Rules\n- One label per mention\n- Label exact text span\n- Do not merge distinct sources\n- Do not invent sources\n- Prioritize precision over recall\n\nYou

In [ ]:
print(messages[0]["content"])

Annotate this text: Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the Excise Tax Act,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the Excise Tax Act. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
Schedule III of the Excise Tax Act. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.
Member W.
Roy Hines W. Roy
Hines
Member
Robert J. Martin Robert J. Martin
Secretary UNOFFICIAL
SUMMARY Appeal
No. 2845 CAN
TRAF

In [ ]:
import torch
def generate(
        assistant,
        messages,  # Full conversation history
        max_new_tokens: int = 512,
    ) -> str:
        """
        Generate a response for Qwen2.5-7B-Instruct.
        
        Args:
            messages: List of dicts with "role" and "content" keys.
                    Example: [
                        {"role": "system", "content": "You are..."},
                        {"role": "user", "content": "Hello"},
                        {"role": "assistant", "content": "Hi!"},
                        {"role": "user", "content": "Another question"}
                    ]
            max_new_tokens: Maximum tokens to generate.
        """
        # Apply chat template - Qwen2.5 handles everything automatically
        text = assistant._format_saul_prompt(messages)
        
        inputs = assistant.tokenizer(
            text,
            return_tensors="pt",
            add_special_tokens=False  # <-- important: we added <s> manually
        ).to(assistant.model.device)
        
        # Recommended sampling parameters for Qwen2.5-Instruct[citation:4][citation:9]
        with torch.no_grad():
            outputs = assistant.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=assistant.temperature,
                do_sample=True,
                repetition_penalty=1.05,
                pad_token_id=assistant.tokenizer.eos_token_id,
            )
        
        # Decode only the newly generated tokens
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response = assistant.tokenizer.decode(generated_ids, skip_special_tokens=True)
        
        return response

In [13]:
generated = assistant.generate(messages=messages)


/home/zagar/myenv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [143]:
generated = generate(assistant=assistant, messages=message)


In [14]:
print(generated)

Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the <legislation>Excise Tax Act</legislation>,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the <legislation>Excise Tax Act</legislation>. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
Schedule III of the <legislation>Excise Tax Act</legislation>. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.
Member W.
Roy Hines W. Roy
Hines
Member
Robert J. Martin Robert J.

In [15]:
corrected_generated_tokens, status = process_output(generated, token_chunk=token_chunk, allowed_labels=allowed_labels, assistant=assistant)


In [16]:
print(decode(corrected_generated_tokens))

 Ottawa, Tuesday,
December 12, 1989 Appeal
No. 2845 IN THE MATTER OF an application heard
May 19, 1989, pursuant to section 51.19 of the <auto_label labelname="legislation">Excise Tax Act</auto_label>,
R.S.C. 1970, c. E-13; AND IN THE MATTER OF a decision of the
Minister of National Revenue dated July 10, 1987, with respect to a notice of
objection filed pursuant to section 51.17 of the <auto_label labelname="legislation">Excise Tax Act</auto_label>. BETWEEN CAN
TRAFFIC SERVICES LTD. Appellant AND THE
MINISTER OF NATIONAL REVENUE Respondent DECISION
OF THE TRIBUNAL The appeal
is dismissed. The Tribunal declares that the sign bridge assemblies installed
by the appellant at the Crowsnest Trail Corridor for the city of Lethbridge,
Alberta, are not bridges within the meaning of paragraph 1(h), Part XII,
Schedule III of the <auto_label labelname="legislation">Excise Tax Act</auto_label>. Sidney
A. Fraleigh Sidney
A. Fraleigh
Presiding
Member Robert
J. Bertrand, Q.C. Robert
J. Bertrand, Q.C.

In [ ]:
user_input =  decode(token_chunk)

message = get_message(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=True)

generated = assistant.generate(message=message)

corrected_generated_tokens, status = process_output(generated, token=token_chunk, allowed_labels=allowed_labels, assistant=assistant)


processed_chunks.append(corrected_generated_tokens)

#### Main processing function

In [49]:
from src.models import get_messages
from tqdm import tqdm

processed_chunks = []
for token_chunk in tqdm(token_chunks):

    user_input =  decode(token_chunk)

    message = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=assistant.has_system_role)

    generated = assistant.generate(messages=message)
    #print("Generated : ", generated)
    corrected_generated_tokens, status = process_output(generated, token_chunk=token_chunk, allowed_labels=allowed_labels, assistant=assistant)
    #print("Corrected generated tokens : ", decode(corrected_generated_tokens))

    processed_chunks.append(corrected_generated_tokens)

100%|██████████| 18/18 [04:03<00:00, 13.53s/it]


In [24]:
import re
import json
from bs4 import BeautifulSoup, NavigableString, Tag

from config import LABEL_SCHEME_PATH


def is_pure_whitespace(node):
    """Return True if node is a NavigableString containing only whitespace."""
    return isinstance(node, NavigableString) and str(node).strip() == ""

def get_significant_children(tag):
    """Return children that are not pure-whitespace text nodes."""
    return [c for c in tag.children if not is_pure_whitespace(c)]

def fix_labels(html_content):

    def normalize_attr_value(k, v):
        if k == "style":
            # Remove all spaces around : and ; for consistent comparison
            return ";".join(
                p.strip() for p in v.replace(" ", "").split(";") if p.strip()
            )
        if isinstance(v, list):
            return " ".join(v)
        return str(v)


    changed = True
    soup = BeautifulSoup(html_content, 'html.parser')

    while changed:
        changed = False
        for label in soup.find_all(["auto_label", "manual_label"]):
            
            # Get the full text of the label (ground truth)
            label_text = label.get_text().strip()
            if not label_text:
                continue

            # For each possible wrapping tag type found inside the label,
            # check if concatenation of all its text == label text
            candidate_tags = {}  # (tag_name, attrs_tuple) -> [list of tag instances]
            for child_tag in label.find_all(True):
                key = (child_tag.name, tuple(sorted(
                    (k, normalize_attr_value(k, v))
                    for k, v in child_tag.attrs.items()
                )))
                if key not in candidate_tags:
                    candidate_tags[key] = []
                candidate_tags[key].append(child_tag)

            # Filter out instances that are descendants of another instance with the same key
            filtered_candidate_tags = {}
            for key, instances in candidate_tags.items():
                top_level = []
                for instance in instances:
                    # Check if any ancestor of this instance is also in the same group
                    is_nested = any(
                        ancestor in instances
                        for ancestor in instance.parents
                    )
                    if not is_nested:
                        top_level.append(instance)
                filtered_candidate_tags[key] = top_level

            winner = None
            winner_instances = None
            for (tag_name, attrs_tuple), instances in filtered_candidate_tags.items():
                # Concatenate text of all instances of this tag
                combined_text = "".join(t.get_text() for t in instances).strip()
                if combined_text.replace(" ", "") == label_text.replace(" ", ""):
                    winner = (tag_name, attrs_tuple)
                    winner_instances = instances
                    break

            if winner is None:
                continue

            winning_tag_name, winning_attrs_tuple = winner
            winning_attrs = {
                k: (v.split(" ") if k == "class" else v)  
                for k, v in winning_attrs_tuple
            }

            # Unwrap all instances of the winning tag inside the label
            for instance in winner_instances:
                instance.unwrap()

            # Wrap the label with the winning tag
            outer = soup.new_tag(winning_tag_name, **winner_instances[0].attrs)
            label.wrap(outer)

            changed = True
            break

    return str(soup)



def clean_html_formatting(html: str, tags_to_clean: set = None, debug: bool = False) -> str:
    """
    Clean HTML by removing useless formatting artifacts WITHOUT changing any text content.
    
    This function performs comprehensive HTML normalization by:
    1. PASS 1: Remove ALL empty tags (tags with NO children)
    2. PASS 2: Merge ALL adjacent identical tags (same name + attributes, no text between)
    3. Repeat until no more changes
    
    CRITICAL: Only merges tags that are truly adjacent with no text nodes between them.
    This preserves ALL text content (including spaces) for character-by-character comparison.
    
    Helps normalize HTML for comparison by removing artifacts like:
    - Empty tags: <i></i>, <span class="..."></span>
    - Adjacent empty + non-empty: <i></i><i>text</i> → <i>text</i>
    - Adjacent identical tags: <b>a</b><b>b</b> → <b>ab</b>
    
    Does NOT merge tags with ANY content between them:
    - <i>a</i> <i>b</i> → stays as is (space preserved)
    - <b>a</b>text<b>b</b> → stays as is
    
    Args:
        html: HTML string to clean
        tags_to_clean: Set of tag names to check. If None, checks common formatting tags.
        debug: If True, print debug information about cleaning operations
    
    Returns:
        Cleaned HTML string with useless formatting removed, all text preserved
    
    Examples:
        >>> clean_html_formatting('<b>text</b><b>more</b>')
        '<b>textmore</b>'
        >>> clean_html_formatting('<i></i><i>text</i>')
        '<i>text</i>'
        >>> clean_html_formatting('<span></span>text')
        'text'
        >>> clean_html_formatting('<i>a</i> <i>b</i>')  # space preserved
        '<i>a</i> <i>b</i>'
    """
    if tags_to_clean is None:
        tags_to_clean = {"span", "i", "b", "strong", "u", "em", "mark", "sup", "sub"}
    
    soup = BeautifulSoup(html, 'html.parser')
    
    max_iterations = 50  # Safety limit
    total_empty_removed = 0
    total_merged = 0
    
    if debug:
        print(f"\n=== Starting clean_html_formatting ===")
        print(f"Tags to clean: {tags_to_clean}")
        print(f"Input length: {len(html)} chars")
    
    # Loop until no more changes can be made
    for iteration in range(max_iterations):
        if debug:
            print(f"\n--- Iteration {iteration + 1} ---")
        
        empty_removed_this_pass = 0
        # PASS 1: Remove ALL empty tags (or whitespace-only tags) in one complete pass
        for tag_name in tags_to_clean:
            while True:
                tags = soup.find_all(tag_name)
                found_empty = False
                
                for tag in tags:
                    children = list(tag.children)
                    has_element_children = any(
                        isinstance(c, Tag) or (isinstance(c, NavigableString) and c.strip() != "")
                        for c in children
                    )
                    
                    # Remove tag if it has no element children (pure text, space, or truly empty)
                    # Always unwrap: keep whatever text is inside, just strip the tag itself
                    if not has_element_children:
                        if debug:
                            print(f"  [PASS 1] Unwrapping <{tag_name}>: {str(tag)[:60]}")
                        tag.unwrap()
                        empty_removed_this_pass += 1
                        found_empty = True
                        break
                
                if not found_empty:
                    break
        
        # PASS 2: Merge ALL adjacent identical tags in one complete pass
        merged_this_pass = 0
        
        for tag_name in tags_to_clean:
            while True:
                tags = soup.find_all(tag_name)
                found_merge = False
                
                for tag in tags:
                    # Look at the next sibling, skipping over pure-whitespace text nodes
                    next_sib = tag.next_sibling
                    whitespace_between = None
                    if (next_sib and
                        isinstance(next_sib, NavigableString) and
                        next_sib.strip() == ""):
                        whitespace_between = next_sib   # remember it so we can remove it
                        next_sib = next_sib.next_sibling

                    # Only merge if next sibling is same tag type with same attributes
                    if (next_sib and
                        hasattr(next_sib, 'name') and
                        next_sib.name == tag_name and
                        dict(tag.attrs) == dict(next_sib.attrs)):

                        if debug:
                            tag_str = str(tag)[:60] + "..." if len(str(tag)) > 60 else str(tag)
                            next_str = str(next_sib)[:60] + "..." if len(str(next_sib)) > 60 else str(next_sib)
                            print(f"  [PASS 2] Merging <{tag_name}> tags:")
                            print(f"           First:  {tag_str}")
                            print(f"           Second: {next_str}")

                        # Merge: move whitespace inside the tag first, then the contents of next_sib
                        if whitespace_between is not None:
                            tag.append(whitespace_between)  # moves the space node inside <b>

                        for child in list(next_sib.children):
                            tag.append(child)

                        next_sib.decompose()
                        merged_this_pass += 1
                        found_merge = True
                        break

                if not found_merge:
                    break
        
        
        total_merged += merged_this_pass
        if debug and merged_this_pass > 0:
            print(f"  [PASS 2] Merged {merged_this_pass} adjacent tag pairs")
        
        # If no changes in this iteration, we're done
        if empty_removed_this_pass == 0 and merged_this_pass == 0:
            if debug:
                print(f"\n=== Cleaning complete after {iteration + 1} iterations ===")
                print(f"Total empty tags removed: {total_empty_removed}")
                print(f"Total tag pairs merged: {total_merged}")
                print(f"Output length: {len(str(soup))} chars")
            break
    
    return str(soup)

def add_attributes_to_auto_labels(html_content: str) -> str:
    """
    Add parent, style, verified, and all label scheme attributes to auto_label tags in HTML content.
    
    The function loads the label scheme from a JSON file to determine:
    - Parent relationships based on nesting context (parent is the immediately enclosing auto_label)
    - Colors for each label (converted to background-color style)
    - All attributes defined in the label scheme for each label
    
    Attributes are initialized as follows:
    - String type: empty string "" (or default value from scheme)
    - Checkbox type: "false" (or default value from scheme)
    - Dropdown type: default value from the label scheme
    
    Also adds verified="false" to all auto_label tags.
    
    Args:
        html_content: HTML string with auto_label tags
    
    Returns:
        Modified HTML string with all attributes added
    """

    with open(LABEL_SCHEME_PATH, 'r', encoding='utf-8') as f:
        label_scheme = json.load(f)
    
    # Build style mapping and attributes mapping from label scheme
    style_map = {}
    attributes_map = {}
    
    # Helper function to determine text color based on background brightness
    def get_text_color(hex_color: str) -> str:
        """Determine if text should be black or white based on background color brightness."""
        # Remove # if present
        hex_color = hex_color.lstrip('#')
        # Convert to RGB
        r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
        # Calculate relative luminance
        luminance = (0.299 * r + 0.587 * g + 0.114 * b) / 255
        return 'black' if luminance > 0.5 else 'white'
    
    # Convert hex color to rgb format
    def hex_to_rgb(hex_color: str) -> str:
        """Convert hex color to rgb() format."""
        hex_color = hex_color.lstrip('#')
        r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
        return f"rgb({r}, {g}, {b})"
    
    # Helper to get attribute value based on type
    def get_attribute_value(attr_config: dict) -> str:
        """Get the initial value for an attribute based on its type."""
        attr_type = attr_config.get('type', 'string')
        if attr_type == 'string':
            return attr_config.get('default', '')
        elif attr_type == 'checkbox':
            default = attr_config.get('default', False)
            return 'true' if default else 'false'
        elif attr_type == 'dropdown':
            return attr_config.get('default', '')
        else:
            return ''
    
    # Process label scheme to build style and attributes mappings
    for parent_label, parent_data in label_scheme.items():
        # Set style for top-level label
        if 'color' in parent_data:
            bg_color = hex_to_rgb(parent_data['color'])
            text_color = get_text_color(parent_data['color'])
            style_map[parent_label] = f"background-color: {bg_color}; color: {text_color};"
        
        # Set attributes for top-level label
        if 'attributes' in parent_data:
            attributes_map[parent_label] = parent_data['attributes']
        
        # Process sublabels
        if 'sublabels' in parent_data:
            for sublabel, sublabel_data in parent_data['sublabels'].items():
                # Set style for sublabel
                if 'color' in sublabel_data:
                    bg_color = hex_to_rgb(sublabel_data['color'])
                    text_color = get_text_color(sublabel_data['color'])
                    style_map[sublabel] = f"background-color: {bg_color}; color: {text_color};"
                
                # Set attributes for sublabel
                if 'attributes' in sublabel_data:
                    attributes_map[sublabel] = sublabel_data['attributes']
    
    # Pattern to match auto_label tags (opening and closing)
    tag_pattern = r'<(/?)auto_label([^>]*)>'
    
    # Stack to track currently open auto_labels
    label_stack = []
    result_parts = []
    last_pos = 0
    
    for match in re.finditer(tag_pattern, html_content, flags=re.IGNORECASE):
        # Add text before this tag
        result_parts.append(html_content[last_pos:match.start()])
        
        is_closing = match.group(1) == '/'
        tag_content = match.group(2)
        
        if is_closing:
            # Closing tag - pop from stack
            if label_stack:
                label_stack.pop()
            result_parts.append(match.group(0))
        else:
            # Opening tag - extract labelname and determine parent
            labelname_match = re.search(r'labelname="([^"]*)"', tag_content)
            if labelname_match:
                labelname = labelname_match.group(1)
                
                # Parent is the labelname of the tag at the top of the stack (or "" if stack is empty)
                parent_value = label_stack[-1] if label_stack else ""
                parent_attr = f'parent="{parent_value}"'
                
                # Get style attribute from label scheme
                style = style_map.get(labelname, '')
                style_attr = f'style="{style}"' if style else ''
                
                # Add verified attribute
                verified_attr = 'verified="false"'
                
                # Get all attributes for this label from label scheme
                label_attrs = attributes_map.get(labelname, {})
                scheme_attrs = []
                for attr_name, attr_config in label_attrs.items():
                    attr_value = get_attribute_value(attr_config)
                    scheme_attrs.append(f'{attr_name}="{attr_value}"')
                
                # Remove existing parent/style/verified/scheme attributes if present
                tag_content = re.sub(r'\s*parent="[^"]*"', '', tag_content)
                tag_content = re.sub(r'\s*style="[^"]*"', '', tag_content)
                tag_content = re.sub(r'\s*verified="[^"]*"', '', tag_content)
                # Remove existing scheme attributes
                for attr_name in label_attrs.keys():
                    tag_content = re.sub(rf'\s*{re.escape(attr_name)}="[^"]*"', '', tag_content)
                
                # Build new tag with all attributes
                all_attrs = [parent_attr, style_attr, verified_attr] + scheme_attrs
                attrs_str = ' '.join(filter(None, all_attrs))  # Filter out empty strings
                new_tag = f'<auto_label{tag_content} {attrs_str}>'
                result_parts.append(new_tag)
                
                # Push this label onto the stack
                label_stack.append(labelname)
            else:
                # No labelname found, keep tag as-is
                result_parts.append(match.group(0))
        
        last_pos = match.end()
    
    # Add remaining text after last tag
    result_parts.append(html_content[last_pos:])
    
    return ''.join(result_parts)


In [25]:
from src.tokenizer_utils import tokenize, decode
from src.html_utils import is_auto_label_tag, is_tag_token

from src.post_processing.token_operations import merge_tokens_general, flatten_token_chunks
from src.post_processing.validation import compare_html_allow_auto_labels
from src.post_processing.bracket_fixing import correct_tokens_brackets, check_tokens_brackets

def tokens_to_html(processed_tokens, html_content):
    """
    Main orchestration function for document-level post-processing.
    
    This function:
    1. Flattens processed chunks from the model into a single token list
    2. Tokenizes the original HTML content
    3. Merges original tokens with processed tokens, preserving auto_label insertions
    4. Validates the merge against the original HTML
    5. Corrects any bracket nesting issues introduced by merging
    6. Validates bracket coherence
    7. Cleans up HTML formatting
    8. Adds label scheme attributes to auto_labels
    
    Args:
        processed_chunks: List of lists of tokens from the model output
        html_content: Original HTML content string
    
    Returns:
        str: Processed HTML with properly annotated auto_label tags
    
    Raises:
        AssertionError: If validation or bracket checking fails
    """
    
    # =====================================================================
    # Step 2: Tokenize original HTML and merge with processed tokens
    # =====================================================================
    original_tokens = tokenize(html_content)
    
    processed_html_content_tokens = merge_tokens_general(
        original_tokens=original_tokens,
        derived_tokens=processed_tokens,
        is_protected_func=lambda tok: is_auto_label_tag(tok) != 0,
        is_opening_protected_func=lambda tok: is_auto_label_tag(tok) == 1,
        is_tag_token_func=lambda tok: is_tag_token(tok),
        log=False
    )
    print(f"   ✓ Merged to {len(processed_html_content_tokens)} tokens")
    
    # =====================================================================
    # Step 4: Validate merge against original HTML
    # =====================================================================
    comparison_result = compare_html_allow_auto_labels(
        decode(processed_html_content_tokens), 
        html_content
    )
    assert comparison_result, (
        "The processed HTML content does not match the original HTML content "
        "when ignoring auto_label tags. Please check the merging and "
        "post-processing steps for errors."
    )
    
    # =====================================================================
    # Step 5: Fix bracket nesting issues
    # =====================================================================
    processed_html_content_tokens_corrected = correct_tokens_brackets(processed_html_content_tokens)
    print(f"   ✓ Corrected {len(processed_html_content_tokens_corrected)} tokens")
    
    # =====================================================================
    # Step 6: Validate bracket coherence
    # =====================================================================
    ok, message, position, context = check_tokens_brackets(processed_html_content_tokens_corrected)
    assert ok, (
        f"The brackets in the merged tokens are not balanced: {message} "
        f"(at position {position}). Please check the merging and bracket "
        f"correction steps for errors.\nContext: {context}"
    )
    print(f"   ✓ {message}")


    # =====================================================================
    # Step 7: Push tags outside auto_labels
    # =====================================================================
    processed_html = decode(processed_html_content_tokens_corrected)
    processed_html_swapped = fix_labels(processed_html)
    
    # =====================================================================
    # Step 8: Clean HTML formatting
    # =====================================================================
    processed_html_cleaned = clean_html_formatting(processed_html_swapped)
    
    # =====================================================================
    # Step 9: Add label scheme attributes
    # =====================================================================
    processed_html_content = add_attributes_to_auto_labels(processed_html_cleaned)
    
    # =====================================================================
    # Final result
    # =====================================================================
    print("\n" + "="*80)
    print("✓ POST-PROCESSING COMPLETE")
    print("="*80)
    print(f"Final HTML length: {len(processed_html_content)} characters\n")
    
    return processed_html_content

#### Post Processing

In [50]:
from src.post_processing.main import tokens_to_html
from src.post_processing.token_operations import flatten_token_chunks

processed_tokens_flat = flatten_token_chunks(processed_chunks)

output_html_content = tokens_to_html(processed_tokens_flat, html_content)

   ✓ Flattened 18 chunks into 9697 tokens
   ✓ Merged to 12123 tokens
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Corrected 12449 tokens
   ✓ Brackets are coherent

✓ POST-PROCESSING COMPLETE
Final HTML length: 97557 characters



#### File saving

In [51]:
output_filename = Path("output") / f"{filename}_0.html"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(output_html_content)

### For DEC1-3 

No chunking needed here, we just need the list of mention already labeled

#### Convert into tokens

In [81]:
from src import extract_body, tokenize, clean_tokens
tokens = tokenize(html_content)


#### Get already extracted mention

In [82]:

from src.extractor import build_processing_segments
from src.extractor import get_list_of_mention
from src.models import get_messages
from tqdm import tqdm
from tqdm import tqdm

parent_mentions = get_list_of_mention(
        tokens=tokens,
        keep_labels=parents,
        label_type="auto_label"  # Process auto_labels from parent extraction
    )

print(f"Found {len(parent_mentions)} parent mentions to process")

segments = build_processing_segments(tokens, parent_mentions)

print(f"Built {len(segments)} token segments "
        f"({sum(s['process'] for s in segments)} to process)")

Found 26 parent mentions to process
Built 53 token segments (26 to process)


#### nesting tag test TBR

In [77]:
html = "blabla blabla <auto_label labelname='legislation'><span lang='EN-CA'><i>Crane Canada Inc. v. Sécurité nationale, compagnie d’assurances and Attorney General of </i></span><i><span lang='EN-CA'>Quebec</span></i></decision></auto_label> ok okokok ok"
soup = BeautifulSoup(html, 'html.parser')
soup = fix_labels(html)
print(soup)

blabla blabla <span lang="EN-CA"><i><auto_label labelname="legislation">Crane Canada Inc. v. Sécurité nationale, compagnie d’assurances and Attorney General of Quebec</auto_label></i></span> ok okokok ok


In [78]:
html = "blabla blabla <auto_label labelname='secondary sources'><i><span lang='EN-GB' style='font-size:12.0pt; letter-spacing:-.1pt'>Webster's Third New International Dictionary of the English Language Unabridged</span></i><span lang='EN-GB' style='font-size:12.0pt;letter-spacing:-.1pt'> (1979)</span></auto_label> ok okokok ok"
soup = BeautifulSoup(html, 'html.parser')
soup = fix_labels(html)
print(soup)

blabla blabla <span lang="EN-GB" style="font-size:12.0pt; letter-spacing:-.1pt"><auto_label labelname="secondary sources"><i>Webster's Third New International Dictionary of the English Language Unabridged</i> (1979)</auto_label></span> ok okokok ok


In [79]:
html = "blabla blabla <auto_label labelname='secondary sources'><span class='MsoFootnoteReference'><span class='MsoFootnoteReference'>[6]</span></span></auto_label> ok okokok ok"
soup = BeautifulSoup(html, 'html.parser')
soup = fix_labels(html)
print(soup)

blabla blabla <span class="MsoFootnoteReference"><span class="MsoFootnoteReference"><auto_label labelname="secondary sources">[6]</auto_label></span></span> ok okokok ok


In [86]:
for segment in segments:
    if not segment["process"]:
        continue
    
    mention = segment["tokens"]
    html_label = segment["meta"]["label"]

    
    prepared_tokens = prepare_label_tokens(mention, input_label_config)
    user_input = decode(prepared_tokens)

    print(user_input)

<legislation>section 51.19 of the <i>Excise Tax Act</i>,
R.S.C. 1970, c. E-13</legislation>
<legislation>section 51.17 of the <i>Excise Tax Act</i></legislation>
<legislation>paragraph 1<i>(h)</i>, Part XII,
Schedule III of the <i>Excise Tax Act</i></legislation>
<legislation>paragraph 1(h), Part XII, Schedule III of the Excise Tax Act</legislation>
<secondary sources>Driedger,
E.A., <u>Construction of Statutes</u> (Second Edition)</secondary sources>
<legislation><i>Excise Tax
Act</i><a href="#_ftn1" name="_ftnref1" title=""><span class="MsoFootnoteReference"><span class="MsoFootnoteReference"><span lang="EN-GB" style='font-size:12.0pt;font-family:"Times New Roman";letter-spacing:-.1pt'>[1]</span></span></span></a>
(the Act)</legislation>
<legislation>LEGISLATION</legislation>
<legislation>27(1) <i>There shall be imposed, levied and
collected a consumption or sales  tax ... on the sale price of all goods</i></legislation>
<legislation>29(1) The tax imposed by section 27 does
not apply

In [93]:
from src.prompts.prompt_utils import build_sublabel_definitions

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

from src.prompts.sublabel_definitions import SUBLABEL_DEFINITIONS_V2


if method in ["DEC1", "DEC2", "DEC3"]:
    sublabels_str = ", ".join(new_labels)
    sublabels_definition = build_sublabel_definitions(set(new_labels) - set(parents), sublabel_definitions=SUBLABEL_DEFINITIONS_V2)

    system_prompt = system_prompt.format(
            sublabels=sublabels_str,
            sublabels_definition=sublabels_definition,
        )

system_prompt used :  decomposed1-3.txt


#### Main processing function

In [83]:
config = LabelTransformConfig(
    use_simplified=False,
    switch_type=False,
    keep_labels=already_labeled_labels,
    keep_attributes=["labelname"]
) # We only remove the attribute
 

failed_count = 0

for idx, segment in enumerate(tqdm(segments, desc="Processing mentions")):
        if not segment["process"]:
            continue


        mention = segment["tokens"]
        html_label = segment["meta"]["label"]

        
        prepared_tokens = prepare_label_tokens(mention, input_label_config)
        user_input = decode(prepared_tokens)

        filtered_fewshot = []
        for example in final_fewshot:
            if example[0].startswith(f"<{html_label.name}>"):
                filtered_fewshot.append(example)
        messages = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=filtered_fewshot, has_system_role=assistant.has_system_role)


        generated = assistant.generate(messages=messages)
        #print(generated)
        
        corrected_generated_tokens, status = process_output(generated=generated, token_chunk=prepare_label_tokens(mention, config), allowed_labels=allowed_labels, assistant=assistant, with_fallback=False)
        #print(decode(corrected_generated_tokens))
        if not status.passed:
            failed_count += 1

        segment["tokens"] = corrected_generated_tokens

processed_tokens = [
        token
        for segment in segments
        for token in segment["tokens"]
    ]

Processing mentions: 100%|██████████| 53/53 [01:13<00:00,  1.39s/it]


#### Post Processing : tokens to HTML

In [41]:
#from src.post_processing.main import tokens_to_html_after_decomposed1_3_prompting
from src.html_utils import is_auto_label_tag
from src.post_processing.html_operations import clean_html_formatting, add_attributes_to_auto_labels
from src.post_processing.validation import compare_html_allow_auto_labels
def tokens_to_html_after_decomposed1_3_prompting(processed_tokens, html_content):
    """
    Orchestration function for converting processed tokens to HTML after the 
    decomposed 1.3 prompting step.
    
    This function:
    1. Validates the processed tokens against the original HTML
    2. Corrects any bracket nesting issues
    3. Validates bracket coherence
    4. Cleans up HTML formatting
    5. Adds label scheme attributes to auto_labels
    
    Args:
        processed_tokens: List of tokens from the model output after decomposed prompting
        html_content: Original HTML content string
    """
    # Useless verification to check if the tokens are the same after processing (except for the auto labels)
    t1 = []
    t2 = []
    for token in processed_tokens:
        if not is_auto_label_tag(token) in [1, 2]:
            t1.append(token)

    original_tokens = tokenize(html_content)
    for token in original_tokens:
        if not is_auto_label_tag(token) in [1, 2]:
            t2.append(token)

    for o, p in zip(t1, t2):
        if o != p:
            print("Original token: ", o)
            print("Processed token: ", p)

    assert t1 == t2, "The tokens are different after processing, which should not happen as we are only adding auto_label tags without changing the original tokens."

    processed_html = decode(processed_tokens)
    comparison_result = compare_html_allow_auto_labels(processed_html, html_content)
    assert comparison_result, "The processed HTML content does not match the original HTML content when ignoring auto_label tags."


    
    processed_html_cleaned = clean_html_formatting(processed_html)

    processed_html_final = add_attributes_to_auto_labels(processed_html_cleaned)


    return processed_html_final


In [84]:
from src.post_processing.main import tokens_to_html_after_decomposed1_3_prompting
processed_html_content = tokens_to_html_after_decomposed1_3_prompting(processed_tokens, html_content)

   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)


#### Save File

In [85]:
output_filename = Path("output") / "test_qwen7b_DEC_fs6_greedy_sentence" / filename / f"{filename}_final.html"
output_filename = Path("output") / f"{filename}_3.html"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(processed_html_content)